# CASE #04 — Yuzey Segmentasyonu — Colab GPU Egitimi

Bu notebook Google Colab'da (Runtime > Change runtime type > GPU) calistirilmak icin hazirlanmistir.
Repo'yu klonlar/kopyalar, Kaggle'dan Severstal veri setini indirir ve `src/train.py` ile egitimi baslatir.

In [ ]:
!nvidia-smi

## 1. Repo'yu getir
Repo GitHub'a pushlanmadiysa, alternatif olarak Drive'a yukleyip mount edebilirsiniz.

In [ ]:
REPO_URL = "https://github.com/nursimaonerr/surface-defect-segmentation.git"
!git clone $REPO_URL project
%cd project

## 2. Bagimliliklar

In [ ]:
!pip install -q -r requirements.txt

## 3. Kaggle API ile veri indirme
Kaggle hesabinizdan **Settings → API → Create New Token** ile aldiginiz token'i asagida girin (Colab Secrets kullanmak isterseniz `KAGGLE_TOKEN` adiyla ekleyip `from google.colab import userdata; userdata.get('KAGGLE_TOKEN')` ile okuyabilirsiniz).

In [ ]:
from google.colab import userdata
import os

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kaggle_token = userdata.get('KAGGLE_TOKEN')  # Colab Secrets'a onceden eklenmis olmali
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(kaggle_token)
os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)

In [ ]:
!bash scripts/download_data.sh

## 4. Egitim
Colab GPU (T4/A100) icin daha buyuk batch-size ve tam epoch sayisi kullanilabilir.

In [ ]:
!python -m src.train \
  --arch unet \
  --encoder resnet34 \
  --epochs 40 \
  --batch-size 16 \
  --lr 1e-4 \
  --checkpoint-dir checkpoints

## 5. Checkpoint'i Drive'a kaydet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/case4_checkpoints
!cp checkpoints/best.pth /content/drive/MyDrive/case4_checkpoints/

## 6. Ornek cikarim ve gorsellestirme

In [ ]:
import glob
sample = glob.glob('data/raw/train_images/*.jpg')[0]
!python -m src.infer --checkpoint checkpoints/best.pth --image "$sample" --out sample_overlay.png
from IPython.display import Image
Image('sample_overlay.png')